In [5]:
from __future__ import annotations

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path(".").resolve()

# -----------------------------
# Inputs
# -----------------------------
# Three figures (each corresponds to one noise setting / chain variant)
FIGS = [
    (
        "iq_chain",
        {
            "Fail rate": "exp04_iq_chain_failrate_over_ideal.csv",
            "MAE overall": "exp04_iq_chain_mean_over_db.csv",
            "MAE only fail": "exp04_iq_chain_mean_over_db_given_over.csv",
        },
    ),
    (
        "chain",
        {
            "Fail rate": "exp04_chain_failrate_over_ideal.csv",
            "MAE overall": "exp04_chain_mean_over_db.csv",
            "MAE only fail": "exp04_chain_mean_over_db_given_over.csv",
        },
    ),
    (
        "realistic_chain",
        {
            "Fail rate": "exp04_realistic_chain_failrate_over_ideal.csv",
            "MAE overall": "exp04_realistic_chain_mean_over_db.csv",
            "MAE only fail": "exp04_realistic_chain_mean_over_db_given_over.csv",
        },
    ),
]

# IJCAI single-column width (inches). If your template differs, set this to \columnwidth.
COL_IN = 3.45
# Height tuned for 1x3 subplots + legend on top
H_IN = 1.70

# Output base name requested
OUT_BASE = "Figure5"

# Per-method linestyle (consistent with your earlier style: v1 dashed)
LINESTYLES = {
    "NE24_v1": "--",
    "NE24_v2": "--",
    "QA-Google_v1": "--",
    "QA-Google_v2": "--",
    "QE-IBM_v1": "--",
    "QE-IBM_v2": "--",
}

# Legend ordering (optional). Any missing methods will be appended.
METHOD_ORDER = [
    "NE24_v1",
    "NE24_v2",
    "QA-Google_v1",
    "QA-Google_v2",
    "QE-IBM_v1",
    "QE-IBM_v2",
    "Agent2",
]

# Marker cycle (no fixed colors; matplotlib default color cycle will be used)
MARKERS = ["o", "s", "^", "v", "D", "P", "X", "+", "*"]

# To keep the same compact IJCAI single-column style as before,
# we only put a y-label on the left subplot. For the other two, the unit is in the title.
TITLE_MAP = {
    "Fail rate": "Fail rate",
    "MAE overall": "MAE overall (dB)",
    "MAE only fail": "MAE only fail (dB)",
}

# -----------------------------
# Plot style (IJCAI-friendly)
# -----------------------------
plt.rcParams.update(
    {
        "font.size": 7,
        "axes.labelsize": 7,
        "axes.titlesize": 7,
        "legend.fontsize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "lines.linewidth": 1,
        "lines.markersize": 1.4,
        "axes.linewidth": 1.0,
        "xtick.major.width": 1.0,
        "ytick.major.width": 1.0,
        "xtick.major.size": 3.5,
        "ytick.major.size": 3.5,
    }
)


def load_wide_csv(path: Path):
    """CSV format: rows=method, columns=x-values (as strings)."""
    df = pd.read_csv(path)

    x_cols = [c for c in df.columns if c != "method"]
    x = sorted(float(c) for c in x_cols)
    col_of = {float(c): c for c in x_cols}

    series = {}
    for _, row in df.iterrows():
        name = row["method"]
        series[name] = [float(row[col_of[v]]) for v in x]

    return x, series


def ordered_methods(methods: list[str]) -> list[str]:
    ordered = [m for m in METHOD_ORDER if m in methods]
    for m in methods:
        if m not in ordered:
            ordered.append(m)
    return ordered


def choose_ticks(x: list[float], max_ticks: int = 6) -> list[float]:
    """Pick up to `max_ticks` ticks evenly across x (keeps endpoints)."""
    if len(x) <= max_ticks:
        return x

    n = len(x)
    if max_ticks <= 2:
        return [x[0], x[-1]]

    raw = [i * (n - 1) / (max_ticks - 1) for i in range(max_ticks)]
    idxs: list[int] = []
    for v in raw:
        j = int(round(v))
        if j not in idxs:
            idxs.append(j)

    # ensure endpoints exist
    if 0 not in idxs:
        idxs = [0] + idxs
    if (n - 1) not in idxs:
        idxs = idxs + [n - 1]

    idxs = sorted(set(idxs))
    return [x[i] for i in idxs]


def format_nl_ticks(ticks: list[float]) -> list[str]:
    # For NL like 0, 0.01, ..., 0.1
    return [f"{t:.2f}" for t in ticks]


def plot_one_figure(noise_name: str, paths_by_metric: dict[str, str], out_base: str):
    metrics = ["Fail rate", "MAE overall", "MAE only fail"]

    # Load all three metrics first
    xs: dict[str, list[float]] = {}
    data: dict[str, dict[str, list[float]]] = {}
    for m in metrics:
        x, series = load_wide_csv(BASE_DIR / paths_by_metric[m])
        xs[m] = x
        data[m] = series

    # Use the first metric's x as the shared x
    x = xs[metrics[0]]

    # Determine a consistent method order
    methods = ordered_methods(list(data[metrics[0]].keys()))

    # Do NOT sharex: we want per-subplot control of the rightmost tick label (avoid overlap between panels).
    fig, axes = plt.subplots(1, 3, figsize=(COL_IN, H_IN), sharey=False)

    for ax_idx, (ax, metric) in enumerate(zip(axes, metrics)):
        series = data[metric]

        for i, method in enumerate(methods):
            if method not in series:
                continue
            ax.plot(
                x,
                series[method],
                marker=MARKERS[i % len(MARKERS)],
                linestyle=LINESTYLES.get(method, "-"),
                label=method,
            )

        # Titles: no (a)(b)(c)
        ax.set_title(TITLE_MAP.get(metric, metric))

        # Keep y-label only on the first subplot (compact single-column layout)
        if ax_idx == 0:
            ax.set_ylabel("Error rate", labelpad=2)
        else:
            ax.set_ylabel("")

        ax.grid(True, linestyle="--", alpha=0.35)

        # x padding
        xr = max(x) - min(x)
        pad = 0.02 * xr if xr > 0 else 1.0
        ax.set_xlim(min(x) - pad, max(x) + pad)

        # ticks for NL
        ticks = choose_ticks(x, max_ticks=6)

        # Remove the rightmost tick label for the first two subplots to avoid overlap
        if (ax_idx < 2) and (len(ticks) > 0) and (abs(ticks[-1] - max(x)) < 1e-9):
            ticks = ticks[:-1]

        ax.set_xticks(ticks)
        ax.set_xticklabels(format_nl_ticks(ticks))

    # Only keep xlabel in the middle subplot (same style as before)
    axes[1].set_xlabel("Noise level")
    axes[0].set_xlabel("")
    axes[2].set_xlabel("")

    # Single legend for the whole figure
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.05),
        ncol=4,
        frameon=True,
        handlelength=1.6,
        columnspacing=1.0,
        borderpad=0.4,
    )

    # Reserve top space for the legend
    fig.subplots_adjust(left=0.11, right=0.995, bottom=0.26, top=0.72, wspace=0.18)

    out_png = BASE_DIR / f"{out_base}_{noise_name}.png"
    out_pdf = BASE_DIR / f"{out_base}_{noise_name}.pdf"
    fig.savefig(out_png, dpi=600, bbox_inches="tight", pad_inches=0.02)
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.close(fig)


def main():
    for noise_name, paths in FIGS:
        plot_one_figure(noise_name, paths, OUT_BASE)


if __name__ == "__main__":
    main()
